In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:

NUM_SEEDS = 20
DIMS = [5, 10, 20 ,30,50]


function = 'ablations_low/gpsample'
function_name = 'low'
function = 'ablations_medium/gpsample'
function_name = 'medium'
function = 'ablations_high/gpsample'
function_name = 'high'



DISTR_PATH = "./Data/"+function+"/local_optima_distribution/"
BEST_HIST_PATH = "./Data/"+function+"/optimizer_history/list_of_bests_"
SAMPLED_POINTS_PATH = "./Data/"+function+"/sampled_data/sampled_data_history_"
LENGTH_SCALE_PATH = "./Data/"+function+"/length_scale/length_scale_"


           
ABLATIONS = [['les_250_8','les_250_4','les_250_16','les_20_8','lesgradcond_20_8'],['les_250_8','les_beta05_250_8','lesGD_250_8','les_20_8','lesCMAES_20_8'],['les_250_8','les_fp_wgrad','localTS']]
ABLATION_NAMES = ['apdx_discrtization_ablation','apdx_difopt_ablations','apdx_other_info_methods']

counter = -1
for METHODS in ABLATIONS: 
    counter += 1

    LABEL_NAMES = {'tracing':'With gradient tracing',
    'mes':'MES',
    'logei':'logEI',
    'turbo':'TURBO',
    'sobol':'Sobol random',
    'std_gibo':'GIBO',
    'hci_gibo':'HCI-GIBO',
    'hci_gibo_09':'HCI-GIBO',
    'les_20_8':'LES-ADAM: L = 20, P = 8',
    'les_250_4':'LES-ADAM: L = 250, P = 4',
    'les_250_16':'LES-ADAM: L = 250, P = 16',
    'les_250_8':'LES-ADAM: L = 250, P = 8 (default)',
    'les_beta05_250_8':'LES-ADAM (beta_1 = 0.5): L = 250, P = 8',
    'les_fp_wgrad':'LES-ADAM-Opt.Cond.: L = 250, P = 8 ',
    'lesGD_250_8':'LES-GD: L = 250, P = 8',
    'lesgradcond_20_8':'LES-ADAM-Grad.-Cond.: L = 20, P = 8',
    'localTS':'Local Thompson Sampling',
    'lesCMAES_20_8':'LES-CMAES: L = 20, P = 8'}
    def decompress_gibo(df):
        data = df[['y','n']].to_numpy(dtype=float)
        repeats = np.diff(data[:,-1].astype(int))
        repeats = np.insert(repeats, 0, data[0,-1])
        repeated = np.repeat(data[:, :-1], repeats, axis=0)
        return np.minimum.accumulate(repeated)




    from matplotlib.lines import Line2D

    avrg_best_history = []
    std_best_history = []
    lower_quantiles_history =[]
    upper_quantiles_history =[]
    avrg_time_deltas = []
    std_time_deltas = []


    for dim in DIMS: 
        data_mean = []
        stds = []

        time_deltas_mean_per_method = []
        time_deltas_std_per_method = []
        lower_quantile =[]
        upper_quantile =[]


        num_objective_calls = min(20*dim, 400)
        for method in range(len(METHODS)):
            y_data = np.zeros((0,0))
            # Collect timestamp differences for all seeds having a timestamp column
            time_arrays = []
            for seed in range(NUM_SEEDS):
                file_identifier = f'{(seed+1):05d}_{dim}_{METHODS[method]}.csv'
                try: 
                    table = pd.read_csv(BEST_HIST_PATH + file_identifier) 
                except: 
                    print(f'Unable to find file {BEST_HIST_PATH+file_identifier}.')
                    continue
                
                if METHODS[method] == 'std_gibo' or METHODS[method] == 'hci_gibo' or METHODS[method] == 'hci_gibo_09' :
                    new_data = decompress_gibo(table.dropna())
                else:
                    new_data = np.reshape(table['f'].to_numpy(), [-1,1])
                if y_data.shape[0] == 0:
                    y_data = new_data[:num_objective_calls, :]
                else: 
                    new_data = new_data[:num_objective_calls, :]
                    y_data = np.concatenate([y_data, new_data], axis=1)

                if 'timestamp' in table.columns:
                    # Crop timestamps to match the # of objective calls
                    timestamps = table['timestamp'].to_numpy()[:num_objective_calls]
                    # Compute the consecutive differences
                    if len(timestamps) > 1:
                        diffs = np.diff(timestamps)
                        time_arrays.append(diffs)


                
            data_mean.append(np.median(y_data, axis=1))
            stds.append(np.std(y_data, axis=1))
            try:
                lower_quantile.append(np.quantile(y_data, 0.25, axis=1))
                upper_quantile.append(np.quantile(y_data, 0.75, axis=1))
            except:
                lower_quantile.append([])
                upper_quantile.append([])
                
            stds.append(np.std(y_data, axis=1))

            # Collect mean/std across seeds for the time-deltas (if available)
            if len(time_arrays) > 0:
                time_arrays = np.array(time_arrays)  # shape: [num_seeds_with_timestamps, num_objective_calls-1]
                time_deltas_mean_per_method.append(np.mean(time_arrays, axis=0))
                time_deltas_std_per_method.append(np.std(time_arrays, axis=0))
            else:
                # No timestamp data for this method in this dimension
                time_deltas_mean_per_method.append(None)
                time_deltas_std_per_method.append(None)

        avrg_best_history.append(data_mean)
        std_best_history.append(stds)
        lower_quantiles_history.append(lower_quantile)
        upper_quantiles_history.append(upper_quantile)

        avrg_time_deltas.append(time_deltas_mean_per_method)
        std_time_deltas.append(time_deltas_std_per_method)
    pass
   
    colors = anonymized
    fig, axs = plt.subplots(2,3, figsize=(14*2/3,6))

    for dim, ax in enumerate(axs.flat[:5]):
        for method in range(len(METHODS)):
            try:
                ax.plot(avrg_best_history[dim][method], color=colors[method], linestyle='-')
                ax.set_title(fr'$d = {DIMS[dim]}$', fontsize=10)
                x = np.arange(1, lower_quantiles_history[dim][method].shape[0]+1)
                
                ax.fill_between(x,lower_quantiles_history[dim][method], upper_quantiles_history[dim][method], 
                                    color=colors[method], alpha=0.1)
                ax.grid(True)
                ax.set_xlabel('Objective function evaluation')
                ax.set_xlim(0,min(20*DIMS[dim], 400))
                #ax.set_xlim(0, len(avrg_best_history[dim][0]))
            except:
                print('error!!!')
                pass

    axs.flat[5].axis('off')
    axs.flat[0].set_yticks(np.linspace(0, -2, 3))
    axs.flat[0].set_ylabel(r'Current best $f$')
    axs.flat[3].set_ylabel(r'Current best $f$')
    #fig.suptitle(f'Optimization history')
    fig.tight_layout()


    legend_elements = [Line2D([0], [0], color=colors[i], lw=1.5, linestyle='-', label=LABEL_NAMES[METHODS[i]]) for i in range(len(METHODS))]

    axs.flat[5].legend(handles=legend_elements, loc='center',frameon=False)

    plt.savefig("plots/ablations/" + ABLATION_NAMES[counter] + "_"+function_name+".pdf", format="pdf", bbox_inches="tight", pad_inches=0.0)
